In [1]:
import numpy as np
import time as time_lib
import random as rd
import copy
import psutil
import gc
import pickle
from qiskit.quantum_info import SparsePauliOp
n_qubit = 1
dim     = 2**n_qubit
nld     = 11
lds     = np.linspace(0,1,num=nld)

In [2]:
def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")

In [3]:
t_x          = 0.5
t_zs         = []
hamiltonians = []
alpha        = 0
for ld in lds:
    t_z = -1 + 2.0 * ld
    h = SparsePauliOp.from_list([('X',t_x),('Z',t_z)])
    t_zs.append(t_z)
    hamiltonians.append(h)
    print(hamiltonians[alpha])
    alpha += 1


SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -1. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 1. +0.j])


In [4]:
hamiltonian_diffs = []
for ild in range(nld-1):
    hamiltonian_diffs.append((hamiltonians[ild+1]-hamiltonians[ild]).simplify())
    print(hamiltonian_diffs[ild])

SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])


In [5]:
hamiltonian_diffs_list = []
for ild in range(nld-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[ild].to_list())
    print(hamiltonian_diffs_list[ild])

[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000018+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]


In [6]:
n_hamiltonians = len(hamiltonians)

In [7]:
# exact eigenvalues
eigen_energies_exact  = np.zeros((nld,dim),dtype=float)
eigen_vectors_exact   = np.zeros((nld,dim,dim),dtype=complex)
for ild in range(nld):
    eigen_e, eigen_v = np.linalg.eigh(hamiltonians[ild].to_matrix())
    indx = np.argsort(eigen_e.real)
    for i in range(dim):
        eigen_energies_exact[ild,i]   = eigen_e[indx[i]].real
        eigen_vectors_exact[ild,:,i] = eigen_v[:,indx[i]]


In [8]:
x_op = SparsePauliOp.from_list([('X',1.0)])
y_op = SparsePauliOp.from_list([('Y',1.0)])
z_op = SparsePauliOp.from_list([('Z',1.0)])
x_mat = x_op.to_matrix()
y_mat = y_op.to_matrix()
z_mat = z_op.to_matrix()

In [9]:
def ComputeUnitaryParams(U):
    theta = 2.0 * np.arccos(np.abs(U[0,0]))
    if (theta<1E-6):
        theta = 2.0*np.arcsin(np.abs(U[0,1]))
    gamma = np.angle(U[0,0])
    if (theta<1E-10):
        phi   = np.angle(U[1,1]/U[0,0])
        lam   = 0
    else:
        phi   = np.angle(U[1,0]/np.sin(theta/2)) - gamma
        lam   = np.angle(-U[0,1]/np.sin(theta/2)) - gamma
    return [theta, phi, lam, gamma]

def ExactTimeEvolution (alpha, time):
    Vl = copy.deepcopy(eigen_vectors_exact[alpha,:,:])
    evol = np.zeros((dim,dim),dtype=complex)
    exp_d = np.diag(np.exp(-1j*eigen_energies_exact[alpha,:]*time))
    evol = Vl@exp_d@Vl.conj().T
    return evol

def TrotterTimeEvolution (alpha, Nt, time):
    dtime  = time/Nt
    dtx = dtime * t_x
    dtz = dtime * t_zs[alpha]
    evol = np.identity(dim)
    for it in range(Nt):
        evol_X = np.cos(dtx)*np.identity(dim) - 1j*np.sin(dtx) * x_mat
        evol_Z = np.cos(dtz)*np.identity(dim) - 1j*np.sin(dtz) * z_mat
        evol = evol_Z@evol
        evol = evol_X@evol
    return evol

In [10]:
# exact results
norms_exact  = np.ones((nld,dim),dtype=float)
for i in range(dim):
    phi = eigen_vectors_exact[0,:,i]
    for alpha in range(1,nld):
        proj_matrix = np.outer(eigen_vectors_exact[alpha,:,i],eigen_vectors_exact[alpha,:,i].conj())
        phi = proj_matrix@phi
        norms_exact[alpha,i] = phi.conj()@phi

x_exps_exact = np.zeros((nld,dim),dtype=float)
y_exps_exact = np.zeros((nld,dim),dtype=float)
z_exps_exact = np.zeros((nld,dim),dtype=float)
for ild in range(nld):
    for i in range(dim):
        x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
        y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
        z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]

/tmp/ipykernel_20350/1490341378.py:8: ComplexWarning: Casting complex values to real discards the imaginary part
  norms_exact[alpha,i] = phi.conj()@phi
/tmp/ipykernel_20350/1490341378.py:15: ComplexWarning: Casting complex values to real discards the imaginary part
  x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_20350/1490341378.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_20350/1490341378.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]


In [11]:
nmc = int(400)


In [12]:
theta_init = np.zeros((dim),dtype=float)
for i in range(dim):
    theta_init[i] = 2.0 * np.angle(eigen_vectors_exact[0,0,i]+ 1j*eigen_vectors_exact[0,1,i])

In [13]:
def Circuit_Initialize(circuit_, i_, qr_):
    # initialize qr_[1:] to i_ th eigenvector
    circuit_.ry(theta_init[i_],qr_[1])

In [14]:
# observable amplitudes
n_obs = 3
# 5 different observables are computed
# 0; norm, 1; dE1, 2; Z
O_timelists         = [[[[] for _ in range(n_obs)] for _ in range(nld)] for _ in range(dim)]

In [15]:
beta = 5

In [16]:
with open('MDH.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

#for i in range(dim):
#    for alpha in range(1,nld):
#        for i_obs in range(n_obs):
#            for imc in range(nmc):
#                times = []
#                for alpha_ in range(1,alpha):
#                    time = rd.gauss(0.0, beta)
#                    times.append(time)
#
#                time = rd.gauss(0.0, beta)
#                times.append(time)
#
#                time = rd.gauss(0.0, beta)
#                times.append(time)
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = rd.gauss(0.0, beta)
#                    times.append(time)
#                O_timelists[i][alpha][i_obs].append(times)
#with open('MDH.time.binary','wb') as file_:
#    pickle.dump([O_timelists],file_)
#

In [17]:
from qiskit import transpile, Aer
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.primitives import Sampler as AerSampler

estimator = AerEstimator()

nshot    = 4000

In [18]:
q_layout = [0,1]
z_hadamard_test = SparsePauliOp.from_sparse_list([('Z',[q_layout[0]],1)],num_qubits=n_qubit+1)
print(z_hadamard_test)

SparsePauliOp(['IZ'],
              coeffs=[1.+0.j])


In [19]:
from qiskit import QuantumRegister
from qiskit import QuantumCircuit, transpile

qr = QuantumRegister(n_qubit+1, 'q')

In [20]:
# parametrized circuit
from qiskit.circuit import ParameterVector
params = ParameterVector('θ',5)

In [21]:
Nt_list = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,30,40,50,60,70,80]
n_Nt = len(Nt_list)
print(n_Nt)

26


In [22]:
# run qzmc;
norms_Nt                = np.ones((n_Nt,n_hamiltonians),dtype=float)
eigen_energies_Nt       = np.zeros((n_Nt,n_hamiltonians),dtype=float)
eigen_energies_Nt[:,0]  = eigen_energies_exact[0,0]

In [23]:
result_values_Nt   = [[[] for _ in range(n_hamiltonians)] for _ in range(n_Nt)]

In [24]:
memory_usage('before run')

[before run] memory usage:    0.18563 GiB


In [25]:
i = 0 # only consider the ground state
for i_Nt in range(n_Nt):
    Nt = Nt_list[i_Nt]
    # real part measurement
    circuit_real = QuantumCircuit(qr,name='-')
    Circuit_Initialize(circuit_real,i,qr)
    circuit_real.h(qr[0])
    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
    circuit_real.p(params[4],qr[0])
    circuit_real.h(qr[0])

    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        params_list = []
        # 0; norm measurement
        i_obs = 0
        for imc in range(nmc):
            U_evol = np.identity(dim,dtype=complex)
            times  = O_timelists[i][alpha][i_obs][imc]
            phase  = 0.0
            i_time = 0
            # construction of P
            for alpha_ in range(1,alpha):
                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                phase += eigen_energies_Nt[i_Nt,alpha_] * time
                i_time += 1

            time = times[i_time]
            U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
            phase += eps * time
            i_time += 1

            # construction of P^\dagger

            time = times[i_time]
            U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
            phase += eps * time
            i_time += 1

            for alpha_ in reversed(range(1,alpha)):
                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                phase += eigen_energies_Nt[i_Nt,alpha_] * time
                i_time += 1

            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
            params_list.append([theta,phi,lam,gamma,phase])
        # 1; dE1 measurement
        i_obs = 1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                    phase += eigen_energies_Nt[i_Nt,alpha_] * time
                    i_time += 1

                # insertion of (H[alpha]-H[alpha-1])
                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P; continued
                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
                phase += eps * time
                i_time += 1

                # construction of P^\dagger

                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                    phase += eigen_energies_Nt[i_Nt,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        # 2; dE2 measurement
        i_obs = 2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                    phase += eigen_energies_Nt[i_Nt,alpha_] * time
                    i_time += 1

                # construction of P; continued
                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
                phase += eps * time
                i_time += 1

                # insertion of (H[alpha+1]-H[alpha])
                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P^\dagger

                time = times[i_time]
                U_evol = TrotterTimeEvolution(alpha,Nt,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = TrotterTimeEvolution(alpha_,Nt,time)@U_evol
                    phase += eigen_energies_Nt[i_Nt,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]

        z_hadamards = [z_hadamard_test]*len(circuits)
        estimator = AerEstimator(run_options={"shots": 4000})
        job = estimator.run(circuits,z_hadamards)
        result_values = job.result().values
        result_values_Nt[i_Nt][alpha] = result_values
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norm += result_values_Nt[i_Nt][alpha][i_meas]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dE1 +=  coeff * result_values_Nt[i_Nt][alpha][i_meas]
                i_meas += 1

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_Nt[i_Nt][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm
        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_Nt[i_Nt,alpha] = eigen_energies_Nt[i_Nt,alpha-1] + dE1
        norms_Nt[i_Nt,alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_Nt[i_Nt,alpha] + dE2
            eps = eps.real

        print(alpha, norms_Nt[i_Nt,alpha], eigen_energies_Nt[i_Nt,alpha]-eigen_energies_exact[alpha,i])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i])
        st = '# {i_Nt}/{n_Nt}: {percent}%'.format(i_Nt=i_Nt+1,n_Nt=n_Nt,percent=((alpha)/(n_hamiltonians-1)*100))

        del job
        del z_hadamards
        del circuits 
        del estimator
        collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_20350/726290858.py:196: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_Nt[i_Nt,alpha] = eigen_energies_Nt[i_Nt,alpha-1] + dE1


1 0.022027500000000037 -0.19003707858588692
1 -0.6133012682868381 -0.7810249675906655
[# 1/26: 10.0%] memory usage:    0.25148 GiB
2 -0.006231249999999989 -1.1632427216933605
2 -1.4201754124535335 -0.6403124237432848
[# 1/26: 20.0%] memory usage:    0.27401 GiB
3 -0.012191249999999987 -1.6274659801926434
3 -2.8870750306148985 -0.5385164807134504
[# 1/26: 30.0%] memory usage:    0.27816 GiB
4 0.01597250000000001 -1.9063162353213974
4 -2.874806107801947 -0.5
[# 1/26: 40.0%] memory usage:    0.27917 GiB
5 -0.0341425 -1.88256428226462
5 -2.5601215788890617 -0.5385164807134505
[# 1/26: 50.0%] memory usage:    0.27822 GiB
6 -0.003580000000000049 -2.2714221032271418
6 -2.193527969415461 -0.640312423743285
[# 1/26: 60.0%] memory usage:    0.28311 GiB
7 -0.004698749999999996 -2.674760504437797
7 -2.972215785323939 -0.7810249675906655
[# 1/26: 70.0%] memory usage:    0.28117 GiB
8 0.02414125 -2.7429331674459023
8 -3.4535496019241565 -0.9433981132056604
[# 1/26: 80.0%] memory usage:    0.28023 Gi

In [26]:
#with open('MDH.Nt.results','rb') as file_:
#    [result_values_Nt] = pickle.load(file_)

with open('MDH.Nt.results','wb') as file_:
    pickle.dump([result_values_Nt],file_)


In [27]:
norms_raw_Nt            = np.zeros((n_Nt,n_hamiltonians,nmc),dtype=float)
dEnorm_raw_Nt           = np.zeros((n_Nt,n_hamiltonians,nmc),dtype=float)

In [28]:
i = 0 # only consider the ground state
for i_Nt in range(n_Nt):
    Nt = Nt_list[i_Nt]
    for alpha in range(1,n_hamiltonians):
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norms_raw_Nt[i_Nt][alpha][imc] = result_values_Nt[i_Nt][alpha][i_meas]
            norm += norms_raw_Nt[i_Nt][alpha][imc]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dEnorm_raw_Nt[i_Nt][alpha][imc] +=  coeff * result_values_Nt[i_Nt][alpha][i_meas]
                i_meas += 1
        for imc in range(nmc):
            dE1 += dEnorm_raw_Nt[i_Nt][alpha][imc]

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_Nt[i_Nt][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm
        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_Nt[i_Nt,alpha] = eigen_energies_Nt[i_Nt,alpha-1] + dE1
        norms_Nt[i_Nt,alpha] = norm

        print(alpha, norms_Nt[i_Nt,alpha], eigen_energies_Nt[i_Nt,alpha]-eigen_energies_exact[alpha,i])
        st = '# {i_Nt}/{n_Nt}: {percent}%'.format(i_Nt=i_Nt+1,n_Nt=n_Nt,percent=((alpha)/(n_hamiltonians-1)*100))

        #collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_20350/990102166.py:23: ComplexWarning: Casting complex values to real discards the imaginary part
  dEnorm_raw_Nt[i_Nt][alpha][imc] +=  coeff * result_values_Nt[i_Nt][alpha][i_meas]


1 0.022027500000000037 -0.19003707858588692
[# 1/26: 10.0%] memory usage:    0.29681 GiB
2 -0.006231249999999989 -1.1632427216933605
[# 1/26: 20.0%] memory usage:    0.29681 GiB
3 -0.012191249999999987 -1.6274659801926434
[# 1/26: 30.0%] memory usage:    0.29681 GiB
4 0.01597250000000001 -1.9063162353213974
[# 1/26: 40.0%] memory usage:    0.29681 GiB
5 -0.0341425 -1.88256428226462
[# 1/26: 50.0%] memory usage:    0.29681 GiB
6 -0.003580000000000049 -2.2714221032271418
[# 1/26: 60.0%] memory usage:    0.29681 GiB
7 -0.004698749999999996 -2.674760504437797
[# 1/26: 70.0%] memory usage:    0.29681 GiB
8 0.02414125 -2.7429331674459023
[# 1/26: 80.0%] memory usage:    0.29681 GiB
9 -0.03416124999999999 -2.617173143423355
[# 1/26: 90.0%] memory usage:    0.29681 GiB
10 -0.028826249999999987 -2.3093253516569274
[# 1/26: 100.0%] memory usage:    0.29681 GiB
1 0.25304875000000027 -0.03478041291104639
[# 2/26: 10.0%] memory usage:    0.29681 GiB
2 0.09758625000000007 -0.16468228311582722
[# 2/2

In [29]:
# compute standard deviations
import random as rd
std_norm    = np.zeros((n_Nt,n_hamiltonians),dtype=float)
std_dE      = np.zeros((n_Nt,n_hamiltonians),dtype=float)
std_E       = np.zeros((n_Nt,n_hamiltonians),dtype=float)
n_boot = 1000
for i_Nt in range(n_Nt):
    for alpha in range(1,n_hamiltonians):
        norm_boot = np.zeros((n_boot),dtype=float)
        dE_boot   = np.zeros((n_boot),dtype=float)
        for i_boot in range(n_boot):
            norm_ = 0.0
            dE_   = 0.0
            for imc in range(nmc):
                jmc = rd.randrange(nmc)
                norm_ += norms_raw_Nt[i_Nt,alpha,jmc]
                dE_ += dEnorm_raw_Nt[i_Nt,alpha,jmc]

            dE_   = dE_/norm_
            norm_ = norm_/nmc
            norm_boot[i_boot] = norm_
            dE_boot[i_boot] = dE_
        norm_boot_mean = 0.0
        dE_boot_mean   = 0.0
        for i_boot in range(n_boot):
            norm_boot_mean += norm_boot[i_boot]
            dE_boot_mean   += dE_boot[i_boot]
        norm_boot_mean /= n_boot
        dE_boot_mean /= n_boot

        var_norm = 0.0
        var_dE = 0.0
        for i_boot in range(n_boot):
            var_norm += (norm_boot[i_boot]-norm_boot_mean)**2
            var_dE += (dE_boot[i_boot]-dE_boot_mean)**2
        var_norm /= (n_boot-1)
        var_dE    /= (n_boot-1)
        std_norm[i_Nt,alpha] = np.sqrt(var_norm)
        std_dE[i_Nt,alpha] = np.sqrt(var_dE)
        std_E[i_Nt,alpha] = std_E[i_Nt,alpha-1] + var_dE
for i_Nt in range(n_Nt):
    for alpha in range(n_hamiltonians):
        std_E[i_Nt,alpha] = np.sqrt(std_E[i_Nt,alpha])

In [30]:
for i_Nt in range(n_Nt):
    print(Nt_list[i_Nt],std_norm[i_Nt,-1],std_E[i_Nt,-1])

1 0.026376461348987745 41.939513263955604
2 0.025933568113736982 19.018350412961976
3 0.0240549720908742 22.700144544707694
4 0.024873373048182226 90.38913826740851
5 0.023824680768687087 0.05550795990580067
6 0.022034914289684418 0.031377006192199366
7 0.0221796651215894 0.02367079827139847
8 0.019277018485094152 0.0181471620647365
9 0.017277422061105517 0.01514797483446833
10 0.015141539755323155 0.012970402055031024
11 0.012762447709545488 0.011733476146577682
12 0.01159821043261018 0.010874767111654627
13 0.010703804544352364 0.0102085963335424
14 0.009720729752379387 0.00978404044244719
15 0.00923103023356123 0.009328165959879151
16 0.00838119228072275 0.009215631441942759
17 0.008457946595056616 0.008946617363036006
18 0.008336890207284838 0.008779455704439615
19 0.007731900218715023 0.008550932674936307
20 0.007721684934817874 0.008420993313239816
30 0.006760246054239749 0.007939259315614074
40 0.0065255668472097095 0.0077747153794011465
50 0.006457578828080165 0.007612169076455

In [31]:
# save to file
with open('Nt.norm.E.save','w') as file_:
    s = '# Nt , norm^2, std(norm^2), Enorm, std(Enorm) '
    s += '\n'
    file_.write(s)
    for i_Nt in range(n_Nt):
        Nt = Nt_list[i_Nt]
        s = '{:}'.format(Nt)
        s += '  {:.16e}  {:.16e}'.format(norms_Nt[i_Nt,-1],std_norm[i_Nt,-1])
        s += '  {:.16e}  {:.16e}'.format(eigen_energies_Nt[i_Nt,-1],std_E[i_Nt,-1])
        print(s)
        s += '\n'
        file_.write(s)

1  -2.8826249999999987e-02  2.6376461348987745e-02  -3.4273593404068223e+00  4.1939513263955604e+01
2  -1.0221250000000022e-02  2.5933568113736982e-02  -7.3183898773790168e-01  1.9018350412961976e+01
3  9.8874999999999918e-03  2.4054972090874201e-02  3.4062995611945329e-01  2.2700144544707694e+01
4  4.7377499999999975e-02  2.4873373048182226e-02  -1.2055991388996032e+00  9.0389138267408512e+01
5  1.7242375000000010e-01  2.3824680768687087e-02  -1.1557546846413267e+00  5.5507959905800673e-02
6  3.0927000000000004e-01  2.2034914289684418e-02  -1.1517625630996180e+00  3.1377006192199366e-02
7  4.0332999999999997e-01  2.2179665121589399e-02  -1.1497126544603022e+00  2.3670798271398470e-02
8  4.9773625000000032e-01  1.9277018485094152e-02  -1.1427633463978764e+00  1.8147162064736499e-02
9  5.6866000000000017e-01  1.7277422061105517e-02  -1.1401310161632652e+00  1.5147974834468329e-02
10  6.2999500000000030e-01  1.5141539755323155e-02  -1.1367823040825216e+00  1.2970402055031024e-02
11  6.77

In [37]:
#i = 0 # only consider the ground state
##for i_rep in range(n_rep):
##for i_rep in range(5):
#for i_rep in range(7,n_rep):
#    rep = rep_list[i_rep]
#    # real part measurement
#    circuit_real = QuantumCircuit(qr,name='-')
#    Circuit_Initialize(circuit_real,i,qr)
#    circuit_real.h(qr[0])
#    for r in range(rep):
#        circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
#        circuit_real.cu(-params[0],-params[2],-params[1],-params[3],qr[0],qr[1])
#    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
#    circuit_real.p(params[4],qr[0])
#    circuit_real.h(qr[0])
#
#    circuit_real = transpile(circuit_real,backend)
#
#    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
#    eps = eps.real
#    for alpha in range(1,n_hamiltonians):
#        params_list = []
#        # 0; norm measurement
#        i_obs = 0
#        for imc in range(nmc):
#            U_evol = np.identity(dim,dtype=complex)
#            times  = O_timelists[i][alpha][i_obs][imc]
#            phase  = 0.0
#            i_time = 0
#            # construction of P
#            for alpha_ in range(1,alpha):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_rep[i_rep,alpha_] * time
#                i_time += 1
#
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            # construction of P^\dagger
#
#            time = times[i_time]
#            U_evol = ExactTimeEvolution(alpha,time)@U_evol
#            phase += eps * time
#            i_time += 1
#
#            for alpha_ in reversed(range(1,alpha)):
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                phase += eigen_energies_rep[i_rep,alpha_] * time
#                i_time += 1
#
#            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#            params_list.append([theta,phi,lam,gamma,phase])
#        # 1; dE1 measurement
#        i_obs = 1
#        nhd1 = len(hamiltonian_diffs[alpha-1])
#        for ihd in range(nhd1):
#            # pass for constant contribution
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                continue
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                # insertion of (H[alpha]-H[alpha-1])
#                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()
#
#                U_evol = Mat@U_evol
#
#                # construction of P; continued
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#
#        # 2; dE2 measurement
#        i_obs = 2
#        if (alpha<n_hamiltonians-1):
#            nhd2 = len(hamiltonian_diffs[alpha])
#        else:
#            nhd2 = 0
#        for ihd in range(nhd2):
#            # pass for constant contribution
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                continue
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                # construction of P; continued
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # insertion of (H[alpha+1]-H[alpha])
#                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()
#
#                U_evol = Mat@U_evol
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#        # 3; Z measurement
#        if (alpha==n_hamiltonians-1):
#            i_obs = 2
#            for imc in range(nmc):
#                U_evol = np.identity(dim,dtype=complex)
#                times  = O_timelists[i][alpha][i_obs][imc]
#                phase  = 0.0
#                i_time = 0
#                # construction of P
#                for alpha_ in range(1,alpha):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                # insertion of Z
#
#                U_evol = z_mat@U_evol
#
#                # construction of P^\dagger
#
#                time = times[i_time]
#                U_evol = ExactTimeEvolution(alpha,time)@U_evol
#                phase += eps * time
#                i_time += 1
#
#                for alpha_ in reversed(range(1,alpha)):
#                    time = times[i_time]
#                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
#                    phase += eigen_energies_rep[i_rep,alpha_] * time
#                    i_time += 1
#
#                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
#                params_list.append([theta,phi,lam,gamma,phase])
#
#        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]
#
#        z_hadamards = [z_hadamard_test]*len(circuits)
#        estimator = AerEstimator(backend_options={
#                "method": "density_matrix",
#                "coupling_map": coupling_map,
#                "noise_model": noise_model,
#                "basis_gates": basis_gates
#            })
#        job = estimator.run(circuits,z_hadamards)
#        result_values = job.result().values
#        result_values_rep[i_rep][alpha] = result_values
#        # compute values
#        norm    = 0.0
#        dE1     = 0.0
#        dE2     = 0.0
#
#        i_meas = 0
#        # 0; norm
#        for imc in range(nmc):
#            norm += result_values_rep[i_rep][alpha][i_meas]
#            i_meas += 1
#        # 1; dE1
#        nhd1 = len(hamiltonian_diffs[alpha-1])
#        for ihd in range(nhd1):
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                continue
#            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
#            for imc in range(nmc):
#                dE1 +=  coeff * result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#
#        # 2; dE2
#        if (alpha<n_hamiltonians-1):
#            nhd2 = len(hamiltonian_diffs[alpha])
#        else:
#            nhd2 = 0
#        for ihd in range(nhd2):
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                continue
#            coeff = hamiltonian_diffs_list[alpha][ihd][1]
#            for imc in range(nmc):
#                dE2 += coeff * result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#
#        dE1     /=norm
#        dE2     /=norm
#
#        # 3; Z
#        if (alpha==n_hamiltonians-1):
#            Znorm = 0.0
#            for imc in range(nmc):
#                Znorm += result_values_rep[i_rep][alpha][i_meas]
#                i_meas += 1
#            Znorm /= norm
#
#        norm    /=nmc
#
#        # add constant contributions
#        for ihd in range(nhd1):
#            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
#                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]
#
#        for ihd in range(nhd2):
#            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
#                dE2 += hamiltonian_diffs_list[alpha][ihd][1]
#
#        eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1
#        norms_rep[i_rep,alpha] = norm
#
#        if (alpha<n_hamiltonians-1):
#            eps = eigen_energies_rep[i_rep,alpha] + dE2
#            eps = eps.real
#        else:
#            Z_rep[i_rep] = Znorm
#
#        print(alpha, norms_rep[i_rep,alpha], eigen_energies_rep[i_rep,alpha]-eigen_energies_exact[alpha,i])
#        if (alpha<n_hamiltonians-1):
#            print(alpha, eps, eigen_energies_exact[alpha+1,i])
#        else:
#            print('# Zdiff: ', Znorm-z_exps_exact[-1,i])
#        st = '# {i_rep}/{n_rep}: {percent}%'.format(i_rep=i_rep+1,n_rep=n_rep,percent=((alpha)/(n_hamiltonians-1)*100))
#
#        del job
#        del z_hadamards
#        del circuits 
#        del estimator
#        collected = gc.collect()
#        memory_usage(st)
#
#
#

/tmp/ipykernel_40757/1109024003.py:256: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_rep[i_rep,alpha] = eigen_energies_rep[i_rep,alpha-1] + dE1


1 0.485556640625 0.0005597158557776316
1 -0.7726427053887395 -0.7810249675906655
[# 8/15: 10.0%] memory usage:    0.78391 GiB
2 0.4862060546875 -0.00020258366330949684
2 -0.6277492685524937 -0.6403124237432848
[# 8/15: 20.0%] memory usage:    0.79208 GiB
3 0.4791259765625 -0.0008876115871233603
3 -0.5142456404259497 -0.5385164807134504
[# 8/15: 30.0%] memory usage:    0.81023 GiB
4 0.465703125 -0.0025095066384307474
4 -0.466449573998417 -0.5
[# 8/15: 40.0%] memory usage:    0.83272 GiB
5 0.440908203125 -0.003151394337639335
5 -0.49844032653990294 -0.5385164807134505
[# 8/15: 50.0%] memory usage:    0.82901 GiB
6 0.41859375 -0.0026434989172086087
6 -0.6173987814223725 -0.640312423743285
[# 8/15: 60.0%] memory usage:    0.85971 GiB
7 0.4214208984375 -0.0028253792388982513
7 -0.7727680763087963 -0.7810249675906655
[# 8/15: 70.0%] memory usage:    0.81748 GiB
8 0.4267626953125 -0.0007382515766871656
8 -0.9367889053723161 -0.9433981132056604
[# 8/15: 80.0%] memory usage:    0.83058 GiB
9 0.

In [38]:
#with open('MDH.rep.results','wb') as file_:
#    pickle.dump([result_values_rep],file_)
